# MMLU-Pro on the chat models of this computer

MMLU-Pro asks multiple-choice questions with ten options in 14 subjects, from law and economics to physics ([Wang, Ma, Zhang, Ni, Chandra, Guo, Ren, Arulraj, He, Jiang, Li, Ku, Wang, Zhuang, Fan, Yue and Chen (2024)](https://doi.org/10.52202/079017-3018)). This notebook does what the page Model Benchmarks does (Compose > Local AI Models > Model Benchmarks...), with the same functions, those of [PollisBenchmarks.jl](https://github.com/Trumpingtons/PollisBenchmarks.jl): it downloads the benchmark, draws a stratified random sample of its questions ([Cochran (1977)](https://openlibrary.org/isbn/9780471162407)), has each chat model chosen answer them, alone in memory, and grades the answers. The runs are kept in `~/.pollis/benchmarks/runs`, where the page shows them too.

In [ ]:
import PollisBenchmarks as PB

## 1. The benchmark

`download` fetches MMLU-Pro from Hugging Face into `~/.pollis/benchmarks/data` the first time, and answers how many questions each subject has.

In [ ]:
info = PB.download("mmlu_pro")
for stratum in info.strata
	println(rpad(stratum.name, 20), stratum.count)
end

## 2. The sample

`draw` takes `size` questions from the subjects named in `strata` (all of them when it is `nothing`), shared equally among them or in proportion to their questions, and drawn at random within each subject with `seed`: the same seed draws the same sample again. With two questions at least in each subject, the score has its 95% interval.

In [ ]:
sample = PB.draw("mmlu_pro"; size = 28, strata = nothing, allocation = "equal", seed = 1)
for stratum in sample.strata
	println(rpad(stratum.name, 20), stratum.size, " of ", stratum.population)
end

## 3. The models

The chat models on this computer. Choose those to run by their numbers.

In [ ]:
# The chat models on this computer, in ~/.pollis/models: tick those to run by their numbers below
installed = sort([joinpath(root, file) for (root, _, files) in walkdir(joinpath(homedir(), ".pollis", "models"))
	for file in files if endswith(file, ".gguf") && !occursin(r"mmproj|embed|rerank"i, file)])
for (number, file) in enumerate(installed)
	println(number, ". ", basename(file))
end

In [ ]:
models = installed[[1]]

## 4. The run

`new_run` keeps the run with its settings: the examples shown before each question (`shots`), the answer worked step by step or alone (`reasoning`), thinking for the models that think, and the most tokens of an answer. `run_models!` then starts each model's llama-server, has it answer the questions of the sample, each graded as it comes, and stops it before the next.

In [ ]:
run = PB.new_run("mmlu_pro", sample; shots = 0, reasoning = true, thinking = false, max_tokens = 2048)
figures = PB.run_models!(run, models; progress = (model, done, total, tokens) -> print("\r", basename(model), ": ", done, " of ", total, "   "))

## 5. The results

The score weights each subject's share of right answers by its share of the questions of the subjects drawn from.

In [ ]:
# Each model's score with its 95% interval, the answers not read or cut off, its speed and its memory
percent(value) = value === nothing ? "-" : string(round(100 * value; digits = 1), "%")
for model in figures
	println(model.label)
	println("  Score ", percent(model.score), model.half === nothing ? "" : " ± " * string(round(100 * model.half; digits = 1)),
		"  (", model.right, " of ", model.questions, " right; ", model.unread, " not read, ", model.cutOff, " cut off)")
	println("  First token ", round(Int, something(model.firstTokenMs, 0)), " ms, writing ",
		round(something(model.writingSpeed, 0); digits = 1), " tokens/s, ", round(something(model.seconds, 0); digits = 1), " s an answer")
	model.memory === nothing || println("  Memory ", round(model.memory / 1024^3; digits = 1), " GB")
end

In [ ]:
# Each model's right answers in each subject
for model in figures, stratum in model.strata
	println(rpad(model.label, 32), rpad(stratum.name, 20), stratum.right, " of ", stratum.asked)
end

In [ ]:
# The first answers, with the answer read from each and whether it is right
for answer in first(PB.load_answers(run), 3)
	println(answer.id, ": read ", something(get(answer, :extracted, nothing), "nothing"), get(answer, :correct, false) ? ", right" : ", wrong")
	println(first(get(answer, :text, get(answer, :error, "")), 600), "\n")
end